# Efficiency, Deployment and Capstone Projects

Notebook for **Chapter 14** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch14.html

Tip: change a number, predict the output, then run the cell again.

## 14.1 Quantization

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
w = np.array([0.31, -0.84, 0.05, 1.29, -0.60, 0.02])
scale = np.abs(w).max() / 127
q = np.clip(np.round(w / scale), -127, 127).astype(np.int8)
wd = q * scale
print("scale", round(scale, 5)); print("int8", q); print("dequantized", wd)
print("abs error", np.abs(w - wd)); print("max error", np.abs(w-wd).max().round(5), " half step", round(scale/2, 5))
s4 = np.abs(w).max() / 7; q4 = np.clip(np.round(w / s4), -7, 7)
print("int4 levels", q4, "dequant", q4*s4, "max err", np.abs(w-q4*s4).max().round(4))

scale 0.01016
int8 [ 31 -83   5 127 -59   2]
dequantized [ 0.3149 -0.8431  0.0508  1.29   -0.5993  0.0203]
abs error [0.0049 0.0031 0.0008 0.     0.0007 0.0003]
max error 0.00488  half step 0.00508
int4 levels [ 2. -5.  0.  7. -3.  0.] dequant [ 0.3686 -0.9214  0.      1.29   -0.5529  0.    ]
max err 0.0814


In [ ]:
w2 = np.array([0.10, -0.12, 0.08, 0.05, -0.07, 8.0]); s2 = np.abs(w2).max()/127
print("with outlier: int codes", np.round(w2/s2), "dequant", np.round(w2/s2)*s2)
rng = np.random.default_rng(0)
W = rng.standard_normal((4, 64)) * np.array([[0.02], [0.05], [0.1], [1.0]])   # rows with very different sizes
def qd(W, axis):
    s = np.abs(W).max(axis=axis, keepdims=True) / 127; return np.round(W/s)*s
print("per-tensor MSE", ((qd(W, None) - W)**2).mean(), "per-row MSE", ((qd(W, 1) - W)**2).mean())
for name, b in [("fp32",4),("fp16",2),("int8",1),("int4",0.5)]: print(name, "7B params ->", 7e9*b/1e9, "GB")

with outlier: int codes [  2.  -2.   1.   1.  -1. 127.] dequant [ 0.126 -0.126  0.063  0.063 -0.063  8.   ]
per-tensor MSE 5.1760703231786286e-05 per-row MSE 1.228009013964982e-05
fp32 7B params -> 28.0 GB
fp16 7B params -> 14.0 GB
int8 7B params -> 7.0 GB
int4 7B params -> 3.5 GB


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
def moons(n, noise=0.15):
    t = rng.uniform(0, np.pi, n//2)
    a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1 - np.cos(t), 0.5 - np.sin(t)]
    X = np.r_[a, b] + noise * rng.standard_normal((n, 2)); y = np.r_[np.zeros(n//2), np.ones(n//2)]
    return X, y
Xtr, ytr = moons(600); Xte, yte = moons(600)
H = 64
W1 = rng.standard_normal((2, H)) * 0.5; b1 = np.zeros(H); W2 = rng.standard_normal((H, 1)) * 0.5; b2 = np.zeros(1)
sig = lambda z: 1/(1+np.exp(-z))
def fwd(X, W1, b1, W2, b2): h = np.maximum(0, X@W1 + b1); return sig(h@W2 + b2)[:, 0], h
lr = 0.5
for ep in range(3000):
    p, h = fwd(Xtr, W1, b1, W2, b2); d = (p - ytr)[:, None] / len(Xtr)
    gW2 = h.T @ d; gb2 = d.sum(0); dh = d @ W2.T * (h > 0); gW1 = Xtr.T @ dh; gb1 = dh.sum(0)
    W1 -= lr*gW1; b1 -= lr*gb1; W2 -= lr*gW2; b2 -= lr*gb2
acc = lambda W1, W2: (((fwd(Xte, W1, b1, W2, b2)[0] > 0.5) == yte).mean())
print("full-precision test accuracy", acc(W1, W2), " params", W1.size + W2.size + H + 1)
def quant(W, bits):
    qmax = 2**(bits-1) - 1; s = np.abs(W).max() / qmax; return np.round(W/s).clip(-qmax, qmax) * s
for bits in (8, 4, 3, 2): print("int%d weights: accuracy" % bits, acc(quant(W1, bits), quant(W2, bits)))

full-precision test accuracy 0.985  params 257
int8 weights: accuracy 0.985
int4 weights: accuracy 0.985
int3 weights: accuracy 0.9866666666666667
int2 weights: accuracy 0.7666666666666667


## 14.2 Pruning and Knowledge Distillation

In [ ]:
# continues the half-moons script in 14.1 (uses acc, W1, W2, H)
def prune(W, frac):
    thr = np.quantile(np.abs(W), frac); return np.where(np.abs(W) < thr, 0, W)
for fr in (0.0, 0.5, 0.8, 0.9, 0.95, 0.99):
    print("prune %.0f%%: accuracy" % (fr*100), acc(prune(W1, fr), prune(W2, fr)), " nonzero", int((prune(W1, fr) != 0).sum() + (prune(W2, fr) != 0).sum()))

prune 0%: accuracy 0.985  nonzero 192
prune 50%: accuracy 0.99  nonzero 96
prune 80%: accuracy 0.9633333333333334  nonzero 39
prune 90%: accuracy 0.6733333333333333  nonzero 20
prune 95%: accuracy 0.5933333333333334  nonzero 11
prune 99%: accuracy 0.5  nonzero 3


In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
def softmax(z): z = z - z.max(); e = np.exp(z); return e / e.sum()
t_logits = np.array([4.0, 1.0, 0.0]); s_logits = np.array([2.0, 1.5, 0.5])
for T in (1, 4):
    print("T", T, "teacher", softmax(t_logits / T), "student", softmax(s_logits / T))
def kl(p, q): return (p * np.log(p / q)).sum()
T = 4; pt, ps = softmax(t_logits/T), softmax(s_logits/T)
soft = T**2 * kl(pt, ps); hard = -np.log(softmax(s_logits)[0])
print("soft loss (T^2 * KL)", round(soft, 4), " hard CE (true class 0)", round(hard, 4))
for alpha in (0.5, 0.9): print("alpha", alpha, "total", round(alpha*soft + (1-alpha)*hard, 4))

T 1 teacher [0.9362 0.0466 0.0171] student [0.5465 0.3315 0.122 ]
T 4 teacher [0.5434 0.2567 0.1999] student [0.3891 0.3434 0.2674]
soft loss (T^2 * KL) 0.7768  hard CE (true class 0) 0.6041
alpha 0.5 total 0.6905
alpha 0.9 total 0.7596


## 14.3 Deployment: ONNX, Runtimes, and Measuring Latency

In [ ]:
# not run here (needs torch and onnx)
import torch
model.eval()                                         # inference mode: batch norm and dropout behave correctly
dummy = torch.randn(1, 3, 224, 224)                  # example input fixes the graph shapes
torch.onnx.export(model, dummy, "model.onnx", input_names=["image"], output_names=["logits"],
                  dynamic_axes={"image": {0: "batch"}})   # allow any batch size
# then, with onnxruntime:
# import onnxruntime as ort; sess = ort.InferenceSession("model.onnx"); out = sess.run(None, {"image": x_numpy})

In [ ]:
import numpy as np, time
rng = np.random.default_rng(0)
W = rng.standard_normal((1024, 1024)).astype(np.float32)
def bench(B, reps=200):
    x = rng.standard_normal((B, 1024)).astype(np.float32)
    for _ in range(10): x @ W                         # warm-up, not counted
    lat = []
    for _ in range(reps):
        t0 = time.perf_counter(); x @ W; lat.append((time.perf_counter() - t0) * 1000)
    lat = np.array(lat); return np.percentile(lat, 50), np.percentile(lat, 95)
for B in (1, 8, 64, 256):
    p50, p95 = bench(B); print("batch %3d  p50 %.3f ms  p95 %.3f ms  throughput %8.0f samples/s  per-sample %.4f ms" % (B, p50, p95, B/(p50/1000), p50/B))

batch   1  p50 0.014 ms  p95 0.015 ms  throughput    71008 samples/s  per-sample 0.0141 ms
batch   8  p50 0.033 ms  p95 0.036 ms  throughput   242270 samples/s  per-sample 0.0041 ms
batch  64  p50 0.101 ms  p95 0.119 ms  throughput   634710 samples/s  per-sample 0.0016 ms
batch 256  p50 0.382 ms  p95 0.421 ms  throughput   670925 samples/s  per-sample 0.0015 ms


## 14.4 Distributed Training: Awareness

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
X = rng.standard_normal((64, 5)); y = X @ np.array([1., -2., 0.5, 3., 0.]) + 0.1*rng.standard_normal(64)
w = rng.standard_normal(5)
grad = lambda Xs, ys, w: 2 * Xs.T @ (Xs @ w - ys) / len(Xs)
full = grad(X, y, w)
workers = [grad(X[i*16:(i+1)*16], y[i*16:(i+1)*16], w) for i in range(4)]   # each "GPU" sees 16 samples
avg = np.mean(workers, axis=0)
print("full-batch gradient     ", full.round(5)); print("average of 4 workers    ", avg.round(5)); print("equal:", np.allclose(full, avg))
print("one worker alone        ", workers[0].round(5))
P = 7e9
bytes_per_param = 2 + 2 + 4 + 4 + 4
print("bytes/param", bytes_per_param, " 7B model:", P*bytes_per_param/1e9, "GB before activations")
for n in (1, 8): print("sharded across", n, "GPUs:", P*bytes_per_param/n/1e9, "GB per GPU")
N = 8; size = 14e9
print("ring all-reduce bytes sent per GPU for 14 GB of gradients on 8 GPUs:", 2*(N-1)/N*size/1e9, "GB")

full-batch gradient      [-0.09289  1.17808 -2.34498 -7.8122  -2.83867]
average of 4 workers     [-0.09289  1.17808 -2.34498 -7.8122  -2.83867]
equal: True
one worker alone         [ 0.46031  1.15345 -3.99136 -3.78263 -4.69383]
bytes/param 16  7B model: 112.0 GB before activations
sharded across 1 GPUs: 112.0 GB per GPU
sharded across 8 GPUs: 14.0 GB per GPU
ring all-reduce bytes sent per GPU for 14 GB of gradients on 8 GPUs: 24.5 GB


In [ ]:
# not run here
import torch, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
dist.init_process_group("nccl")
rank = dist.get_rank(); torch.cuda.set_device(rank)
model = DDP(model.cuda(rank), device_ids=[rank])
loader = torch.utils.data.DataLoader(ds, batch_size=32, sampler=DistributedSampler(ds))
# in the loop: sampler.set_epoch(epoch); loss.backward() now all-reduces gradients automatically

## 14.5 Capstone Projects

## 14.6 Interview Bank (Chapters 1 to 14)

Practise answering aloud in 60 seconds each. Answers are model answers; add your own project examples.

## 14.7 Worked Memory Budgets: DDP, ZeRO and FSDP

Section 14.4 said that FSDP and DeepSpeed ZeRO "shard" training state across GPUs. Here is the arithmetic, so you can answer "will this fit?" before you rent the hardware. Recall the 16 bytes per parameter for mixed-precision Adam training: 2 (fp16 weights) + 2 (fp16 gradients) + 12 (optimizer state: fp32 master copy 4, Adam first moment 4, second moment 4).

In [ ]:
N, P = 8, 7e9
rows = [("no sharding (DDP)", 2+2+12), ("ZeRO-1 / optimizer sharded", 2+2+12/N),
        ("ZeRO-2 / + gradients sharded", 2+(2+12)/N), ("ZeRO-3 / FSDP full shard", 16/N)]
for name, b in rows:
    print(f"{name:30s} {b:6.2f} bytes/param  {b*P/1e9:7.2f} GB per GPU")

no sharding (DDP)               16.00 bytes/param   112.00 GB per GPU
ZeRO-1 / optimizer sharded       5.50 bytes/param    38.50 GB per GPU
ZeRO-2 / + gradients sharded     3.75 bytes/param    26.25 GB per GPU
ZeRO-3 / FSDP full shard         2.00 bytes/param    14.00 GB per GPU


## 14.8 Post-Training and Alignment: SFT, Reward Models and DPO

In [ ]:
import numpy as np
loss = np.array([2.0, 1.5, 0.9, 0.5, 0.4])   # per-token cross-entropy
is_response = np.array([0, 0, 1, 1, 1])
print("mean over all tokens     :", round(loss.mean(), 4))
print("mean over response tokens:", round(loss[is_response==1].mean(), 4))

mean over all tokens     : 1.06
mean over response tokens: 0.6


In [ ]:
import numpy as np
sig = lambda z: 1/(1+np.exp(-z))
# reward model, Bradley-Terry
r_w, r_l = 2.1, 0.4
print("reward model loss:", round(-np.log(sig(r_w-r_l)),4), " P(w preferred) =", round(sig(r_w-r_l),4))
# DPO
beta = 0.1
def dpo(pw, pl, rw, rl):
    margin = beta*((pw-rw)-(pl-rl))
    return margin, -np.log(sig(margin)), -sig(-margin)   # d loss / d margin
for name, args in [("start (policy = reference)", (-13.0,-14.0,-13.0,-14.0)),
                   ("good update", (-12.0,-15.0,-13.0,-14.0)),
                   ("wrong direction", (-14.0,-13.0,-13.0,-14.0))]:
    m, L, g = dpo(*args)
    print(f"{name:28s} margin {m:+.2f}  loss {L:.4f}  dL/dmargin {g:+.4f}")

reward model loss: 0.1678  P(w preferred) = 0.8455
start (policy = reference)   margin +0.00  loss 0.6931  dL/dmargin -0.5000
good update                  margin +0.20  loss 0.5981  dL/dmargin -0.4502
wrong direction              margin -0.20  loss 0.7981  dL/dmargin -0.5498


## Chapter recap

## Practice set

**1.** (easy) Quantize `w = 0.5` with `s = 0.01`. What is `q` and the dequantized value? **Answer:** `q = round(0.5/0.01) = 50`; dequantized `50 × 0.01 = 0.5`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch14.html